# Literały w C++

Materiały, ćwiczenia i quiz o literałach w C++: wbudowanych, surowych i definiowanych przez użytkownika.

## Czym są literały?

**Literał** to wartość zapisana wprost w kodzie źródłowym: `42`, `3.14`, `'a'`, `"Ala"`, `true`, `nullptr`.
Standard opisuje literały jako tokeny programu, które reprezentują stałe wartości osadzone w kodzie
([cppreference: Literals](https://en.cppreference.com/cpp/language/expressions#Literals)).
Każdy literał ma **typ**, a o typie decyduje sam zapis: system liczbowy, przedrostek (np. `u8`, `L`, `R`)
i przyrostek, czyli **sufiks** (np. `u`, `f`, `s`, `ms`, `_kib`).

Literały w C++ dzielą się na trzy grupy — i na nich opiera się całe laboratorium:

| Grupa | Przykłady | Typ (przykłady) | Od |
|---|---|---|---|
| wbudowane: liczbowe, znakowe, napisowe, logiczne, wskaźnikowe | `42`, `0xFF`, `1'000'000`, `2.5f`, `'x'`, `"tekst"`, `true`, `nullptr` | `int`, `float`, `char`, `const char[6]`, `bool`, `std::nullptr_t` | C++98; `nullptr` — C++11; `0b…` i separator `'` — C++14 |
| surowe napisy (*raw string literals*) | `R"(C:\dane\plik.txt)"` | jak zwykły napis, ale bez sekwencji ucieczki | C++11 |
| zdefiniowane przez użytkownika, w tym sufiksy biblioteki standardowej | `"tekst"s`, `250ms`, `4_kib` | `std::string`, `std::chrono::milliseconds`, dowolny typ | C++11; sufiksy biblioteki — C++14 (`sv` — C++17) |

Literał zdefiniowany przez użytkownika (*user-defined literal*, UDL) to zwykły literał z sufiksem, za którym
stoi funkcja — **operator literałowy** `operator""_sufiks`. Kompilator zamienia `4_kib` na wywołanie
`operator""_kib(4)`, więc literał może zwrócić obiekt dowolnego typu, np. odległość z jednostką
([cppreference: User-defined literals](https://en.cppreference.com/cpp/language/user_literal)).

```cpp
#include <chrono>
#include <cstddef>
#include <iostream>
#include <string>

using namespace std::literals;  // standard suffixes: s, sv, h, min, ms, ...

constexpr std::size_t operator""_kib(unsigned long long n) { return n * 1024; }

int main() {
    auto count = 42;                 // int
    auto path = R"(C:\dane\plik.txt)";  // const char*, backslashes kept as they are
    auto name = "Ala"s;              // std::string
    auto timeout = 250ms;            // std::chrono::milliseconds
    auto buffer = 4_kib;             // std::size_t, 4096

    std::cout << count << ' ' << path << ' ' << name << ' '
              << timeout.count() << ' ' << buffer << '\n';
    // 42 C:\dane\plik.txt Ala 250 4096
}
```

Notebooki prowadzą od literałów wbudowanych do własnych:

1. **01 — literały wbudowane**: typy literałów liczbowych, znakowych i napisowych, surowe napisy oraz
   sufiksy biblioteki standardowej dla `std::string`, `std::string_view` i `std::chrono`.
2. **02 — literały definiowane przez użytkownika**: własne operatory literałowe, ich dozwolone formy,
   wybór przeciążenia i typy z jednostkami.
3. **03 — literały w czasie kompilacji**: literały `constexpr` i `consteval`, które sprawdzają poprawność
   wartości już podczas kompilacji, także dla napisów.

## Dlaczego to ważne?

**Typ literału decyduje o wyniku.** `1 / 2` to dzielenie całkowite (wynik `0`), a `1.0 / 2` daje `0.5`;
`auto s = "Ala";` to wskaźnik `const char*`, a `auto s = "Ala"s;` — obiekt `std::string`. Kto nie zna typu
literału, ten nie wie, która funkcja zostanie wywołana.

**Czytelność.** Separatory cyfr (C++14) pozwalają pisać `1'000'000` zamiast `1000000`, a surowe napisy —
wyrażenie regularne `R"(\d+\.\d+)"` zamiast `"\\d+\\.\\d+"`. Dokumentacja Microsoftu wprost zaleca surowe napisy
dla wyrażeń regularnych z klasami znaków oraz dla tekstów HTML i XML
([Microsoft Learn: Raw string literals](https://learn.microsoft.com/en-us/cpp/cpp/string-and-character-literals-cpp#raw-string-literals-c11)).

**Jednostki.** W 1999 roku NASA straciła sondę *Mars Climate Orbiter*. Komisja badająca wypadek ustaliła,
że przyczyną było użycie w oprogramowaniu naziemnym danych o pracy silników w jednostkach anglosaskich zamiast
metrycznych
([raport komisji](https://llis.nasa.gov/llis_lib/pdf/1009464main1_0641-mr.pdf), s. 6,
[komunikat JPL](https://www.jpl.nasa.gov/news/mars-climate-orbiter-team-finds-likely-cause-of-loss)).
Jeden program podawał siłę w funtach, drugi oczekiwał niutonów
([Wikipedia: Mars Climate Orbiter — Utrata sondy](https://pl.wikipedia.org/wiki/Mars_Climate_Orbiter#Utrata_sondy)).
Liczba typu `double` nie niesie informacji o jednostce. C++ Core Guidelines pokazują, jak tę informację
zapisać w kodzie: zamiast `change_speed(2.3)` — `change_speed(23_m / 10s)`, gdzie jednostka jest częścią typu
i kompilator odrzuci wywołanie bez niej
([Core Guidelines P.1](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#rp-direct)).

**Równe prawa typów użytkownika.** FAQ C++11 Bjarne'a Stroustrupa uzasadnia literały użytkownika tym, że
w C++98 typy użytkownika nie miały literałów, co łamało zasadę, by były wspierane tak dobrze jak typy wbudowane
([C++11 FAQ: User-defined literals](https://isocpp.org/wiki/faq/cpp11-language#udls)).

## Najważniejsze zasady

1. **Zapis literału wyznacza jego typ.** `42` to `int`, `42u` to `unsigned int`, `4.2` to `double`,
   `4.2f` to `float`, a `"abc"` to tablica `const char[4]` (z kończącym znakiem `'\0'`).
2. **Sufiks może dać typ z biblioteki.** Sufiksy standardowe nie mają podkreślenia (`s`, `ms`, `sv`);
   sufiksy własne **muszą** zaczynać się od `_` — pozostałe są zarezerwowane dla biblioteki standardowej
   ([cppreference: Literal operators](https://en.cppreference.com/cpp/language/user_literal#Literal_operators)).
3. **Literał użytkownika to wywołanie funkcji.** Operator literałowy może więc zwrócić dowolny typ i — jako
   `constexpr` albo `consteval` — zostać obliczony, a nawet sprawdzony, już podczas kompilacji.

## Cele

Po tym laboratorium potrafisz:

* odczytać typ literału wbudowanego z jego zapisu (system liczbowy, separatory, przedrostek, sufiks);
* użyć surowego napisu tam, gdzie zwykły wymagałby wielu `\\` (ścieżki, wyrażenia regularne);
* korzystać z literałów biblioteki standardowej: `std::string`, `std::string_view`, `std::chrono`;
* napisać własny operator literałowy w formie całkowitej, zmiennoprzecinkowej i napisowej, zwracający typ
  z jednostką;
* wyjaśnić, kiedy literał `constexpr` jest liczony podczas kompilacji, i użyć `consteval`, aby błędna
  wartość literału była błędem kompilacji.

## Jak korzystać z notebooków

* Komórkę uruchamiasz skrótem `Shift+Enter` (albo przyciskiem ▶ na pasku narzędzi).
* Komórki z kodem zaczynające się od `!` wykonują polecenia powłoki — w tym laboratorium
  głównie `cmake` i `ctest`. Każda taka komórka jest niezależna: zaczyna w katalogu notebooka.
* Fragmenty kodu C++ w tekście są ilustracją — każdy z nich był skompilowany i uruchomiony (GCC i Clang),
  a wynik podajemy w komentarzu. Możesz je skopiować do własnego pliku albo do [Compiler Explorer](https://godbolt.org/).
* Pliki do edycji otwierasz kliknięciem w link w treści ćwiczenia (otworzą się w edytorze JupyterLab).
  **Zapisz plik** (`Ctrl+S`) przed uruchomieniem komórki, która go kompiluje.

## Wymagania

| Narzędzie | Minimalna wersja |
|---|---|
| Kompilator C++20 | GCC 11, Clang 14; Visual Studio 2022 (kompilator MSVC 19.30+) i Apple Clang 15 (Xcode 15) — bez gwarancji |
| CMake | 3.22 |
| Python + JupyterLab | Python 3.10, JupyterLab 4 |
| Internet | tylko przy pierwszej kompilacji — CMake pobiera GoogleTest 1.18.0 (raz, do `lab/_deps`) |

Minimalne wersje kompilatorów wynikają z funkcji C++20 używanych w laboratorium, przede wszystkim `consteval`
([cppreference: wsparcie C++20 w kompilatorach](https://en.cppreference.com/cpp/compiler_support/20#P1073R3)).
CMake 3.22 i Ubuntu 22.04 to zarazem minimum GoogleTest 1.18 według
[polityki wsparcia Google](https://github.com/google/oss-policies-info/blob/main/foundational-cxx-support-matrix.md).
Laboratorium zostało sprawdzone na GCC 11.4 i Clang 14.0 (Ubuntu 22.04, CMake 3.22) oraz na GCC 16.2
i Clang 23.1; MSVC i Apple Clang nie były testowane. Domyślne pakiety w Ubuntu 22.04/24.04 i Debianie 12/13
spełniają wymagania.

## Sprawdzenie środowiska

Poniższe komórki budują mały projekt testowy z katalogu [`_smoke/`](_smoke/smoke_test.cpp).
Używa on tych samych ustawień co wszystkie ćwiczenia (plik [`cmake/lab.cmake`](cmake/lab.cmake)):
standard C++20, GoogleTest i gMock pobrane przez `FetchContent` w przypiętej wersji.

Każde ćwiczenie budujemy tymi samymi trzema krokami:

1. `cmake -S <źródła> -B <katalog build>` — konfiguracja (za pierwszym razem pobiera i konfiguruje GoogleTest),
2. `cmake --build <katalog build>` — kompilacja,
3. `ctest --test-dir <katalog build>` — uruchomienie wszystkich testów.

Opcje `--config Debug` i `-C Debug` są potrzebne tylko w Visual Studio (generator wielokonfiguracyjny);
na Linuksie i macOS nic nie zmieniają.

In [ ]:
# Tool versions
!cmake --version

In [ ]:
# Configure, build and run the smoke test
!cmake -S _smoke -B _smoke/build && cmake --build _smoke/build --config Debug --parallel && ctest --test-dir _smoke/build -C Debug --output-on-failure

Na końcu powinieneś zobaczyć podsumowanie `ctest` z informacją, że przeszły wszystkie testy, np.:

```
100% tests passed, 0 tests failed out of 2
```

(nowsze wersje CMake skracają je do `100% tests passed out of 2`).

Jeśli konfiguracja się nie powiedzie, sprawdź wersję CMake i kompilatora. Komunikat o braku
`<compare>` albo błąd przy `operator<=>` oznacza zbyt stary kompilator.

## Plan laboratorium

1. [**01. Literały wbudowane**](01_builtin/01_builtin.ipynb) — typy literałów, surowe napisy, `std::literals`.
2. [**02. Literały definiowane przez użytkownika**](02_udl/02_udl.ipynb) — `operator""`, typy z jednostkami; ćwiczenia 1–3.
3. [**03. Literały w czasie kompilacji**](03_compile_time/03_compile_time.ipynb) — `constexpr`, `consteval`, literały napisowe; ćwiczenia 4–5.

Notebooki najlepiej przerabiać po kolei: każdy korzysta z poprzedniego. Na końcu każdego są ciekawostki dla
dociekliwych — można je pominąć bez szkody dla ćwiczeń.

---

[następny: 01. Literały wbudowane →](01_builtin/01_builtin.ipynb)